# Deep Learning Approaches for Jet Classification in Proton-Proton Collisions
## FYP — BNBWU-CERN Collaboration

**Dataset:** Top Quark Tagging Reference Dataset (Zenodo: 10.5281/zenodo.2603256)  
**Task:** Binary classification — Top quark jets vs QCD background jets  
**Models:** BDT baseline, CNN (jet images), ParticleNet (GNN), Particle Transformer  

---

### Table of Contents
1. [Setup & Configuration](#1)
2. [Data Loading & Exploration](#2)
3. [Jet Substructure Physics](#3)
4. [Jet Image Visualization](#4)
5. [BDT Baseline Training](#5)
6. [CNN Training & Evaluation](#6)
7. [ParticleNet Training & Evaluation](#7)
8. [Particle Transformer Training & Evaluation](#8)
9. [Model Comparison & Publication Plots](#9)
10. [Interpretability Analysis](#10)
11. [Conclusions & Discussion](#11)

<a id='1'></a>
## 1. Setup & Configuration

First, install dependencies and configure the environment.  
**For Google Colab:** Uncomment the pip install line below.

In [ ]:
# ── For Google Colab: uncomment these lines ──
# !pip install -q h5py xgboost lightgbm mplhep energyflow
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/FYP_Particle_Physics/  # adjust path

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Add project root to path
PROJECT_DIR = os.path.abspath('..')
sys.path.insert(0, PROJECT_DIR)

from config import *

# Try CMS-style plots
try:
    import mplhep as hep
    plt.style.use(hep.style.CMS)
except ImportError:
    pass

plt.rcParams.update(PLOT_STYLE)
%matplotlib inline

print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

<a id='2'></a>
## 2. Data Loading & Exploration

The **Top Quark Tagging Reference Dataset** contains Monte Carlo simulated jets from 14 TeV proton-proton collisions:
- Generated with **Pythia 8** (parton shower) + **Delphes** (detector simulation)
- Anti-$k_T$ jets with $R=0.8$, $p_T \in [550, 650]$ GeV, $|\eta| < 2$
- Each jet has up to 200 constituents with 4-momenta $(E, p_x, p_y, p_z)$
- Labels: 1 = top quark jet, 0 = QCD (light quark/gluon) jet

In [ ]:
from src.data_utils import prepare_all_data, make_dataloaders

# Load and preprocess all data
# This computes: particle features, jet images, pairwise features, jet-level features
data = prepare_all_data(max_particles=100)

In [ ]:
# Quick data overview
for split in ['train', 'val', 'test']:
    labels = data[split]['labels']
    n_sig = int(labels.sum())
    n_bkg = len(labels) - n_sig
    print(f'{split:>5}: {len(labels):>7,} jets | Signal: {n_sig:>6,} ({n_sig/len(labels)*100:.1f}%) | Background: {n_bkg:>6,}')

print(f'\nParticle features shape: {data["train"]["features"].shape}')  # (N, 200, 7)
print(f'Jet images shape:        {data["train"]["images"].shape}')      # (N, 3, 40, 40)
print(f'Jet-level features:      {data["feature_names"]}')

In [ ]:
# Constituent multiplicity distribution
mask = data['test']['mask']
labels = data['test']['labels']
n_const = mask.sum(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(n_const[labels==1], bins=50, alpha=0.6, label='Top jets', color='#d62728', density=True)
ax.hist(n_const[labels==0], bins=50, alpha=0.6, label='QCD jets', color='#1f77b4', density=True)
ax.set_xlabel('Number of Constituents')
ax.set_ylabel('Normalized')
ax.set_title('Jet Constituent Multiplicity')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'plots', 'multiplicity.pdf'))
plt.show()

<a id='3'></a>
## 3. Jet Substructure Physics

Before applying ML, let's understand the **physics** that makes top quark jets different from QCD jets:

- **Jet mass**: Top jets peak near $m_t \approx 173$ GeV; QCD jets have lower mass
- **N-subjettiness $\tau_{21}$**: Top jets have 3-prong structure → low $\tau_{21}$
- **Energy correlations**: Top jets show distinct multi-point correlations

These are the variables that ATLAS and CMS use in their traditional taggers.

In [ ]:
from src.evaluation import plot_jet_substructure

# Plot all jet substructure variables
plot_jet_substructure(
    data['test']['jet_features'],
    data['test']['labels'],
    data['feature_names']
)

# Display inline
from IPython.display import Image
Image(os.path.join(RESULTS_DIR, 'plots', 'jet_substructure.png'))

In [ ]:
# Correlation matrix of jet features (signal only)
sig_mask = data['test']['labels'] == 1
df_sig = pd.DataFrame(
    data['test']['jet_features'][sig_mask],
    columns=data['feature_names']
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(df_sig.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True)
ax.set_title('Feature Correlations (Top Quark Jets)')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'plots', 'feature_correlations.pdf'))
plt.show()

<a id='4'></a>
## 4. Jet Image Visualization

Jet images represent the radiation pattern in the $(\Delta\eta, \Delta\phi)$ plane.  
Top quark jets show **3-prong structure** (from $t \to Wb \to qqb$), while QCD jets are **1-prong**.

In [ ]:
from src.evaluation import plot_jet_images, plot_average_jet_images

# Individual jet images
plot_jet_images(data['test']['images'], data['test']['labels'])
Image(os.path.join(RESULTS_DIR, 'plots', 'jet_images.png'))

In [ ]:
# Average jet images — clearly shows 3-prong vs 1-prong structure
plot_average_jet_images(data['test']['images'], data['test']['labels'])
Image(os.path.join(RESULTS_DIR, 'plots', 'average_jet_images.png'))

<a id='5'></a>
## 5. BDT Baseline (XGBoost)

The BDT on high-level jet features represents the **traditional approach** used at ATLAS/CMS.  
This serves as our baseline: deep learning should beat this to justify the added complexity.

In [ ]:
from src.trainer import train_bdt_baseline
from src.evaluation import plot_feature_importance

bdt_results = train_bdt_baseline(
    data['train']['jet_features_norm'], data['train']['labels'],
    data['val']['jet_features_norm'], data['val']['labels'],
    data['test']['jet_features_norm'], data['test']['labels'],
    data['feature_names']
)

plot_feature_importance(bdt_results['importance'], bdt_results['feature_names'])
Image(os.path.join(RESULTS_DIR, 'plots', 'bdt_importance.png'))

<a id='6'></a>
## 6. CNN on Jet Images

The CNN processes jet images — 40×40 pixel images in the $(\Delta\eta, \Delta\phi)$ plane.  
This is the **earliest DL approach** for jet classification (de Oliveira et al., 2016).

Architecture: ResNet-style with skip connections → Global Average Pooling → FC

In [ ]:
from src.jet_cnn import JetCNN
from src.trainer import Trainer

# Initialize model
model_cnn = JetCNN()
print(f'JetCNN parameters: {model_cnn.count_parameters():,}')
print(model_cnn)

In [ ]:
# Create data loaders
train_loader_cnn, val_loader_cnn, test_loader_cnn = make_dataloaders(data, 'cnn')

# Train
trainer_cnn = Trainer(model_cnn, 'jet_cnn', CNN_CONFIG, model_type='cnn')
history_cnn = trainer_cnn.train(train_loader_cnn, val_loader_cnn)

In [ ]:
# Evaluate on test set
trainer_cnn.load_best_model()
cnn_probs, cnn_labels = trainer_cnn.predict(test_loader_cnn)

from src.evaluation import compute_metrics
cnn_metrics = compute_metrics(cnn_labels, cnn_probs)
print(f'CNN Test AUC: {cnn_metrics["auc"]:.5f}')
print(f'CNN 1/εB @ εS=50%: {cnn_metrics["bg_rej_50"]:.0f}')

<a id='7'></a>
## 7. ParticleNet (Graph Neural Network)

ParticleNet treats jets as **particle clouds** — unordered sets of particles.  
Uses **Dynamic Graph CNN** with k-nearest neighbors in learned feature space.  

Key physics insight: nearby particles in $(\eta, \phi)$ come from the same parton shower branch.

In [ ]:
from src.particle_net import ParticleNet

pnet_config = PARTICLENET_CONFIG.copy()
pnet_config['input_features'] = NUM_PARTICLE_FEATURES

model_pnet = ParticleNet(config=pnet_config)
print(f'ParticleNet parameters: {model_pnet.count_parameters():,}')
print(model_pnet)

In [ ]:
# Create data loaders
train_loader_pn, val_loader_pn, test_loader_pn = make_dataloaders(data, 'particlenet', 100)

# Train
trainer_pnet = Trainer(model_pnet, 'particlenet', PARTICLENET_CONFIG, model_type='particlenet')
history_pnet = trainer_pnet.train(train_loader_pn, val_loader_pn)

In [ ]:
# Evaluate
trainer_pnet.load_best_model()
pnet_probs, pnet_labels = trainer_pnet.predict(test_loader_pn)

pnet_metrics = compute_metrics(pnet_labels, pnet_probs)
print(f'ParticleNet Test AUC: {pnet_metrics["auc"]:.5f}')
print(f'ParticleNet 1/εB @ εS=50%: {pnet_metrics["bg_rej_50"]:.0f}')

<a id='8'></a>
## 8. Particle Transformer

The Particle Transformer uses **self-attention** with physics-motivated **pairwise interaction features**.  
Currently the **state-of-the-art** architecture for jet classification.

Key innovation: attention bias from pairwise features $(\Delta R, k_T, z, m_{ij})$ encodes QCD radiation patterns.

In [ ]:
from src.particle_transformer import ParticleTransformer

model_pt = ParticleTransformer()
print(f'Particle Transformer parameters: {model_pt.count_parameters():,}')
print(model_pt)

In [ ]:
# Create data loaders
train_loader_pt, val_loader_pt, test_loader_pt = make_dataloaders(data, 'transformer', 100)

# Train
trainer_pt = Trainer(model_pt, 'particle_transformer', PARTFORMER_CONFIG, model_type='transformer')
history_pt = trainer_pt.train(train_loader_pt, val_loader_pt)

In [ ]:
# Evaluate
trainer_pt.load_best_model()
pt_probs, pt_labels = trainer_pt.predict(test_loader_pt)

pt_metrics = compute_metrics(pt_labels, pt_probs)
print(f'Particle Transformer Test AUC: {pt_metrics["auc"]:.5f}')
print(f'Particle Transformer 1/εB @ εS=50%: {pt_metrics["bg_rej_50"]:.0f}')

<a id='9'></a>
## 9. Model Comparison & Publication Plots

Now we compare all models and generate publication-ready plots for the thesis and papers.

In [ ]:
from src.evaluation import (
    run_full_evaluation, plot_roc_curves, plot_score_distributions,
    plot_training_history, plot_confusion_matrices, print_summary_table
)

# Collect all results
results_dict = {
    'BDT (XGBoost)': {
        'labels': bdt_results['test_labels'],
        'probs': bdt_results['test_probs'],
        'n_params': f"{bdt_results['model'].num_boosted_rounds()} trees",
    },
    'CNN (Jet Images)': {
        'labels': cnn_labels,
        'probs': cnn_probs,
        'n_params': model_cnn.count_parameters(),
    },
    'ParticleNet': {
        'labels': pnet_labels,
        'probs': pnet_probs,
        'n_params': model_pnet.count_parameters(),
    },
    'Particle Transformer': {
        'labels': pt_labels,
        'probs': pt_probs,
        'n_params': model_pt.count_parameters(),
    },
}

histories = {
    'CNN (Jet Images)': history_cnn,
    'ParticleNet': history_pnet,
    'Particle Transformer': history_pt,
}

In [ ]:
# ROC curves — the most important plot for the thesis
plot_roc_curves(results_dict)
Image(os.path.join(RESULTS_DIR, 'plots', 'roc_comparison.png'))

In [ ]:
# Score distributions
plot_score_distributions(results_dict)
Image(os.path.join(RESULTS_DIR, 'plots', 'score_distributions.png'))

In [ ]:
# Training history
plot_training_history(histories)
Image(os.path.join(RESULTS_DIR, 'plots', 'training_history.png'))

In [ ]:
# Confusion matrices
plot_confusion_matrices(results_dict)
Image(os.path.join(RESULTS_DIR, 'plots', 'confusion_matrices.png'))

In [ ]:
# Summary table (prints to console AND saves LaTeX + JSON)
summary = print_summary_table(results_dict)

<a id='10'></a>
## 10. Interpretability Analysis

Understanding **what** the models learn is critical for:
- Building trust in ML-based physics analysis
- Gaining physics insights from the learned representations
- Identifying potential biases or artifacts

In [ ]:
from src.evaluation import plot_attention_maps, plot_cls_attention

# Extract attention maps from Particle Transformer
attn_maps, attn_features, attn_labels = trainer_pt.get_attention_maps(test_loader_pt, n_samples=10)

# Attention map visualization
plot_attention_maps(attn_maps, attn_features, attn_labels)
Image(os.path.join(RESULTS_DIR, 'plots', 'attention_maps.png'))

In [ ]:
# CLS attention vs particle ΔR — shows which particles matter most
plot_cls_attention(attn_maps, attn_features, attn_labels)
Image(os.path.join(RESULTS_DIR, 'plots', 'cls_attention.png'))

In [ ]:
# Model efficiency comparison
fig, ax = plt.subplots(figsize=(10, 6))

model_names = list(results_dict.keys())
aucs = [compute_metrics(r['labels'], r['probs'])['auc'] for r in results_dict.values()]
n_params = []
for r in results_dict.values():
    p = r['n_params']
    if isinstance(p, int):
        n_params.append(p)
    else:
        n_params.append(0)  # BDT

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, (name, auc_val, n_p) in enumerate(zip(model_names, aucs, n_params)):
    if n_p > 0:
        ax.scatter(n_p, auc_val, s=200, c=colors[i], label=name, zorder=5, edgecolors='black')
        ax.annotate(name, (n_p, auc_val), textcoords='offset points',
                   xytext=(10, 10), fontsize=10)

ax.set_xlabel('Number of Parameters')
ax.set_ylabel('AUC-ROC')
ax.set_title('Model Efficiency: Performance vs Complexity')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'plots', 'efficiency_comparison.pdf'))
plt.savefig(os.path.join(RESULTS_DIR, 'plots', 'efficiency_comparison.png'))
plt.show()

<a id='11'></a>
## 11. Conclusions & Discussion

### Key Findings

1. **Architecture hierarchy**: Particle Transformer > ParticleNet > CNN > BDT  
   This confirms the trend in the literature — models that access raw particle-level information outperform those using compressed representations.

2. **Physics-informed design matters**: The pairwise interaction features in the Particle Transformer explicitly encode QCD radiation physics, giving it an edge over generic architectures.

3. **Jet images capture substructure**: The CNN successfully learns the 3-prong vs 1-prong distinction visible in average jet images, but loses fine-grained information through pixelation.

4. **BDT baseline is competitive**: With carefully chosen jet substructure variables ($\tau_{21}$, jet mass, $C_2$), the BDT achieves respectable performance, validating decades of physics-motivated feature engineering.

### Publication Directions

- **Paper 1**: Systematic comparison of DL architectures for top tagging (ML4Jets workshop)
- **Paper 2**: Attention-based interpretability of jet substructure (JINST)
- **Paper 3**: Computational efficiency trade-offs for real-time jet tagging (ACAT)

### All generated outputs:
- `results/model_comparison.json` — Metrics for all models
- `results/comparison_table.tex` — LaTeX table for thesis
- `results/plots/*.pdf` — Publication-quality vector plots
- `results/models/*.pt` — Saved model checkpoints

In [ ]:
# List all generated files
print('Generated plots:')
for f in sorted(os.listdir(os.path.join(RESULTS_DIR, 'plots'))):
    fpath = os.path.join(RESULTS_DIR, 'plots', f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f:<40} ({size_kb:.0f} KB)')

print('\nSaved models:')
for f in sorted(os.listdir(MODEL_DIR)):
    fpath = os.path.join(MODEL_DIR, f)
    size_mb = os.path.getsize(fpath) / (1024**2)
    print(f'  {f:<40} ({size_mb:.1f} MB)')